# Reporting Multiple Measures Example

This notebook provides an example way to structure your code when running multiple models and comparing across multiple evaluation measures. 

## Prepare Initial Data

In [1]:
import pandas as pd
import os

diabetes_file_path = os.path.join(os.getcwd(), 'datasets', 'Diabetes_and_LifeStyle_Dataset.csv')
diabetes_df = pd.read_csv(diabetes_file_path)

diabetes_response_var = diabetes_df['diagnosed_diabetes']
# For now, we will take numeric variables only
numeric_vars = ['Age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day',
       'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp',
       'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol',
       'ldl_cholesterol', 'triglycerides', 'glucose_fasting',
       'glucose_postprandial', 'insulin_level', 'hba1c',
       'diabetes_risk_score']

diabetes_predictor_vars = diabetes_df[numeric_vars]

## Definitions for pipeline

In [ ]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# This function takes in a collection of predictors and creates a train-test split
def create_train_test_split(X_base, y_base):
    X_base_with_constant = sm.add_constant(X_base)
    X_train, X_test, y_train, y_test = train_test_split(X_base_with_constant, y_base, test_size=0.8)
    return [X_train, X_test, y_train, y_test]

# This function creates a logistic regression on the provided training data
def create_logit_regression(y_train, X_train):
    logit_model = sm.Logit(y_train, X_train)
    logit_regression = logit_model.fit()
    return logit_regression

# This function outputs a dictionary of measure results based on test and prediction data
def return_evaluation_measures(y_test, y_pred):
    eval_dict = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1 score': f1_score(y_test, y_pred)
    }
    return eval_dict

# This function is our pipeline for creating a logistic regression based on some X set
def report_eval_dict_by_X(X_base, y_base, cutoff_prob):
    X_train, X_test, y_train, y_test = create_train_test_split(X_base, y_base)
    logit_regression = create_logit_regression(y_train, X_train)
    y_pred_probs = logit_regression.predict(X_test)
    y_pred = (y_pred_probs >= cutoff_prob).astype(int)
    eval_dict_for_X = return_evaluation_measures(y_test, y_pred)
    return eval_dict_for_X

## Predictor Variable Sets

In [ ]:
# We define some subsets of the full dataset to create logistic regressions on
X_full_model = diabetes_predictor_vars
X_some_model = diabetes_df[['glucose_fasting', 'hba1c', 'diabetes_risk_score']]
X_19_model = diabetes_df[['glucose_fasting']]
X_22_model = diabetes_df[['hba1c']]
X_23_model = diabetes_df[['diabetes_risk_score']]

## Create Logistic Regressions and Report Results

In [ ]:
list_of_eval_measures = []
for X_set in [X_full_model, X_some_model, X_19_model, X_22_model, X_23_model]:
    # We define a cutoff prob to use for all models
    cutoff_prob = 0.5
    temp_dict = report_eval_dict_by_X(X_set, diabetes_response_var, cutoff_prob)
    list_of_eval_measures.append(temp_dict)

# Creates a DataFrame to organize our results
eval_df = pd.DataFrame(list_of_eval_measures)
# Adds the names of the models to the first column of the DataFrame
eval_df.insert(0, 'model name', ['full model', 'v19, v22, v23', 'v19', 'v22', 'v23'])

Optimization terminated successfully.
         Current function value: 0.343774
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.344430
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.517378
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.356621
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.628845
         Iterations 5


In [16]:
eval_df

,model name,accuracy,precision,recall,f1 score
0,full model,0.861700,0.879862,0.891618,0.885701
1,"v19, v22, v23",0.859233,0.874961,0.893459,0.884113
2,v19,0.731995,0.763805,0.802125,0.782496
3,v22,0.848994,0.860537,0.892542,0.876248
4,v23,0.632827,0.648293,0.843241,0.733027
